In [5]:
# -*- coding: utf-8 -*-
"""0804國數(四則)互動遊戲.ipynb (語音簡潔 + 防重疊版)"""

import random
import time
import json
import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# ==========================================
# 🌟 讓網頁瀏覽器開口說話的魔法函數 (終極發音修正版)
# ==========================================
def speak(text):
    """利用網頁瀏覽器的語音合成功能唸出中文"""

    # 偷偷把容易唸錯的破音字換成同音字，只影響聲音，不影響畫面顯示！
    safe_text = text.replace('乘', '成')

    js_code = f"""
    <script>
    window.speechSynthesis.cancel(); // 🌟 新增：說話前先清空佇列，避免重複堆疊碎碎唸
    var msg = new SpeechSynthesisUtterance('{safe_text}');
    msg.lang = 'zh-TW'; // 設定為台灣中文語音
    msg.rate = 1.2;     // 語速
    window.speechSynthesis.speak(msg);
    </script>
    """
    display(HTML(js_code))

# ==========================================
# 0. 自動建立題庫檔案 (確保檔案永遠存在)
# ==========================================
questions_data = [
    {"q": "下列哪一個詞語的國字完全「正確」？", "options": ["1. 己經", "2. 已經", "3. 以經", "4. 已京"], "ans": "2", "hint1": "這個詞代表事情發生過了。注意左邊是「己」還是「已」。", "hint2": "「已經」的「已」是半開口喔！"},
    {"q": "「因為」的「為」注音應該怎麼念？", "options": ["1. ㄨㄟˊ", "2. ㄨㄟˋ", "3. ㄨㄟ", "4. ㄨㄟˇ"], "ans": "2", "hint1": "表示原因的時候，通常念四聲喔！", "hint2": "正確讀音是 ㄧㄣ ㄨㄟˋ。"},
    {"q": "「他『ㄐㄧㄢ』持要去動物園。」引號中的國字應該是哪一個？", "options": ["1. 堅", "2. 艱", "3. 尖", "4. 煎"], "ans": "1", "hint1": "表示心意確定不改變，下面有一個「土」字。", "hint2": "正確的詞語是「堅持」。"},
    {"q": "下列哪一個字的部首和「樹」一樣？", "options": ["1. 花", "2. 草", "3. 林", "4. 葉"], "ans": "3", "hint1": "「樹」是木部，選項中哪一個也是木部呢？", "hint2": "兩個木加起來的字就是了！"},
    {"q": "「告訴」的「訴」注音是什麼？", "options": ["1. ㄕㄨˋ", "2. ㄙㄨˋ", "3. ㄘㄨˋ", "4. ㄕㄨ"], "ans": "2", "hint1": "是平舌音（沒有捲舌）喔！", "hint2": "發音是四聲的 ㄙㄨˋ。"},
    {"q": "請找出句子裡的錯別字：「天上飄著美麗的白雲，天氣真晴郎。」", "options": ["1. 飄", "2. 麗", "3. 晴", "4. 郎"], "ans": "4", "hint1": "天氣很好叫做「晴ㄌㄤˇ」，選項中的字是不是少了一個部首？", "hint2": "應該是「晴朗」，右邊少了一個「月」。"},
    {"q": "「想」的部首是什麼？", "options": ["1. 木", "2. 目", "3. 心", "4. 相"], "ans": "3", "hint1": "思考和感情通常和哪一個器官有關？看看這個字的下面。", "hint2": "是「心」部喔！"},
    {"q": "下列注音哪一個正確？「操」場", "options": ["1. ㄔㄠ", "2. ㄘㄠ", "3. ㄘㄠˋ", "4. ㄔㄠˋ"], "ans": "2", "hint1": "是平舌音（不捲舌），而且是一聲。", "hint2": "正確注音是 ㄘㄠ。"},
    {"q": "「興高采烈」的「興」注音是什麼？", "options": ["1. ㄒㄧㄥ", "2. ㄒㄧㄥˋ", "3. ㄒㄧㄣ", "4. ㄒㄧㄣˋ"], "ans": "2", "hint1": "這是一個破音字，在這裡表示心情很激動、開心。", "hint2": "四聲，唸作「ㄒㄧㄥˋ」。"},
    {"q": "下列哪個字加上「氵」(水部) 後，會變成另外一個正確的字？", "options": ["1. 羊", "2. 去", "3. 可", "4. 以上三個都可以"], "ans": "4", "hint1": "試著把它們加上水部：洋、法、河。", "hint2": "這三個字加上水部後，都是我們學過的字喔！所以答案是 4。"}
]

with open('questions.json', 'w', encoding='utf-8') as f:
    json.dump(questions_data, f, ensure_ascii=False, indent=4)

# ==========================================
# 0.5 注入 CSS 美化樣式 (加入注音字型支援)
# ==========================================
css_style = widgets.HTML("""
<style>
/* 嘗試套用電腦內建的注音字型，字體稍微放大以利閱讀 */
* {
    font-family: '王漢宗中仿宋注音', '王漢宗中明注音', '華康楷書體W3(P)注音', 'DFBopomofo', sans-serif !important;
    font-size: 105%;
}

.menu-box {
    background-color: #ffffff !important;
    border-radius: 15px !important;
    box-shadow: 0 4px 15px rgba(0,0,0,0.08) !important;
    border: 1px solid #eaeaea !important;
}
.game-box {
    background-color: #f8f9fa !important;
    border-radius: 15px !important;
    padding: 20px !important;
    border: 1px solid #eaeaea !important;
}
.game-btn, .game-btn button {
    border-radius: 20px !important;
    font-weight: bold !important;
    box-shadow: 0 2px 4px rgba(0,0,0,0.05) !important;
}
</style>
""")
display(css_style)


# ==========================================
# 1. 建立基礎遊戲類別 (BaseGame)
# ==========================================
class BaseGame:
    def __init__(self, title, max_questions=10, game_type="遊戲", menu_cb=None):
        self.title = title
        self.max_questions = max_questions
        self.game_type = game_type
        self.menu_cb = menu_cb

        self.score = 0
        self.question_count = 0
        self.is_answered = False
        self.start_time = 0

        self._setup_ui()
        self._bind_events()

    def _setup_ui(self):
        self.title_html = widgets.HTML(value=f"<h2 style='color: #2c3e50; border-bottom: 2px solid #3498db; padding-bottom: 10px;'>🎮 {self.title}</h2><p style='color: #555;'>請在下方輸入答案<b>（可直接按 Enter 鍵送出）</b>，如果不會可以先點擊下方的提示喔！</p>")
        self.question_html = widgets.HTML(value="<h3>請等候題目載入...</h3>")

        button_layout = widgets.Layout(width='120px', height='35px')
        self.answer_input = widgets.Text(value="", placeholder="請輸入", description="你的答案:", layout=widgets.Layout(width='180px'))

        self.submit_btn = widgets.Button(description="送出答案", button_style="success", icon="check", layout=button_layout)
        self.score_btn = widgets.Button(description="📊 目前得分", button_style="warning", layout=button_layout)
        self.end_btn = widgets.Button(description="🛑 結束遊戲", button_style="danger", layout=button_layout)
        self.restart_btn = widgets.Button(description="🔄 重新開始", button_style="primary", layout=button_layout)
        self.back_btn = widgets.Button(description="🏠 關閉遊戲", button_style="info", layout=button_layout)

        self.hint1_btn = widgets.Button(description="💭 提示 1", button_style="info", layout=widgets.Layout(width='250px', height='35px'))
        self.hint2_btn = widgets.Button(description="💭 提示 2", button_style="info", layout=widgets.Layout(width='250px', height='35px'))
        self.ans_btn = widgets.Button(description="✅ 顯示參考答案", button_style="warning", layout=widgets.Layout(width='250px', height='35px'))

        for btn in [self.submit_btn, self.score_btn, self.end_btn, self.restart_btn, self.back_btn, self.hint1_btn, self.hint2_btn, self.ans_btn]:
            btn.add_class('game-btn')

        self.out_main = widgets.Output()
        self.out_hint1 = widgets.Output()
        self.out_hint2 = widgets.Output()
        self.out_ans = widgets.Output()

    def _bind_events(self):
        self.submit_btn.on_click(self.check_answer)
        self.answer_input.on_submit(self.check_answer)
        self.end_btn.on_click(self.end_game)
        self.score_btn.on_click(self.show_score)
        self.restart_btn.on_click(self.restart_game)
        self.back_btn.on_click(self.return_to_menu)
        self.hint1_btn.on_click(self.show_hint1)
        self.hint2_btn.on_click(self.show_hint2)
        self.ans_btn.on_click(self.show_ans)

    def get_ui(self):
        ui_layout = widgets.VBox([
            self.title_html,
            self.question_html,
            widgets.HBox([self.answer_input, self.submit_btn], layout=widgets.Layout(margin='10px 0')),
            widgets.HBox([self.end_btn, self.score_btn, self.restart_btn, self.back_btn], layout=widgets.Layout(margin='10px 0')),
            widgets.VBox([self.hint1_btn, self.out_hint1, self.hint2_btn, self.out_hint2, self.ans_btn, self.out_ans]),
            self.out_main
        ])
        return ui_layout

    def start_game(self):
        self.score = 0
        self.question_count = 0
        self.is_answered = False
        self.start_time = time.time()
        self._clear_all_outputs()
        speak("遊戲開始！")
        self.generate_question()

    def restart_game(self, b=None):
        self.answer_input.disabled, self.submit_btn.disabled, self.end_btn.disabled = False, False, False
        self.score_btn.disabled, self.hint1_btn.disabled, self.hint2_btn.disabled, self.ans_btn.disabled = False, False, False, False
        self.start_game()

    def end_game(self, b=None):
        end_time_timestamp = time.time()
        elapsed_seconds = int(end_time_timestamp - self.start_time)
        self.question_html.value = f"<h2 style='color: #d35400;'>🏆 遊戲結束！</h2><h3 style='color: #34495e;'>總結算：你挑戰了 {self.question_count} 題，總共答對了 {self.score} 題！<br>⏳ 總共花費了 {elapsed_seconds} 秒</h3>"

        try:
            current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            record_str = f"紀錄時間: {current_time} | 題型: {self.game_type} | 總題數: {self.question_count} | 答對題數: {self.score} | 耗時: {elapsed_seconds} 秒\n"
            with open('/content/score_history.txt', 'a', encoding='utf-8') as f:
                f.write(record_str)
        except Exception as e:
            pass

        with self.out_main:
            clear_output()
            if self.score == self.max_questions and self.max_questions > 0:
                print("💯 太神啦！全對，你是滿分的小天才！")
                speak("太神啦！全對！")
            elif self.score >= self.max_questions / 2:
                print("👍 表現得很棒喔！多加練習一定能拿滿分！")
                speak("表現得很棒喔！")
            else:
                print("💪 沒關係，多練習就會變強！繼續加油！")
                speak("繼續加油喔！")

    def show_score(self, b=None):
        with self.out_main:
            clear_output()
            print(f"📊 【即時戰況】\n🔹 目前題號：第 {self.question_count} / {self.max_questions} 題\n🔹 目前得分：{self.score} 分")

    def return_to_menu(self, b=None):
        if self.menu_cb:
            self.menu_cb()

    def _clear_all_outputs(self):
        with self.out_main: clear_output()
        with self.out_hint1: clear_output()
        with self.out_hint2: clear_output()
        with self.out_ans: clear_output()

    def generate_question(self): pass
    def check_answer(self, b=None): pass
    def show_hint1(self, b=None): pass
    def show_hint2(self, b=None): pass
    def show_ans(self, b=None): pass


# ==========================================
# 2. 數學遊戲類別 (MathGame)
# ==========================================
class MathGame(BaseGame):
    def __init__(self, title, operator='+', max_questions=10, menu_cb=None):
        type_map = {'+': "加法遊戲", '-': "減法遊戲", '*': "乘法遊戲", '/': "除法遊戲"}
        game_type = type_map.get(operator, "數學遊戲")
        super().__init__(title, max_questions, game_type, menu_cb)
        self.operator = operator
        self.current_a = 0
        self.current_b = 0
        self.asked_questions = set()

    def start_game(self):
        self.asked_questions.clear()
        super().start_game()

    def generate_question(self):
        if self.question_count >= self.max_questions:
            self.end_game()
            return

        while True:
            if self.operator == '-':
                a, b = random.randint(1, 18), random.randint(1, 9)
                if a < b: a, b = b, a
            elif self.operator == '/':
                b = random.randint(1, 9)
                ans = random.randint(1, 9)
                a = b * ans
            else:
                a, b = random.randint(1, 9), random.randint(1, 9)

            check_pair = (a, b)
            if check_pair not in self.asked_questions:
                self.current_a, self.current_b = a, b
                self.asked_questions.add(check_pair)
                break

        self.question_count += 1
        self.is_answered = False

        display_operator = self.operator
        if self.operator == '*': display_operator = '×'
        if self.operator == '/': display_operator = '÷'

        self.question_html.value = f"<h3 style='color: #2980b9;'>▶️ 第 {self.question_count} / {self.max_questions} 題： {self.current_a} {display_operator} {self.current_b} = ?</h3>"
        self.answer_input.value = ""
        self._clear_all_outputs()
        with self.out_main: print("✨ 新題目來囉！請輸入答案。")

        # 🌟 修正發音：把「乘」改為「乘以」，發音才會正確
        spoken_op = {'+': '加', '-': '減', '*': '乘以', '/': '除以'}[self.operator]
        speak_text = f"第{self.question_count}題，{self.current_a} {spoken_op} {self.current_b} 等於多少？"
        speak(speak_text)

    def check_answer(self, b=None):
        with self.out_main:
            clear_output()
            if self.is_answered: return
            user_input = self.answer_input.value.strip()

            if not user_input.isdigit():
                print("⚠️ 哎呀，你還沒輸入答案，或是輸入了不對的文字喔！請填入正確數字。")
                speak("請輸入數字")
                return

            self.is_answered = True
            if self.operator == '+': correct_ans = self.current_a + self.current_b
            elif self.operator == '-': correct_ans = self.current_a - self.current_b
            elif self.operator == '*': correct_ans = self.current_a * self.current_b
            elif self.operator == '/': correct_ans = self.current_a // self.current_b

            if int(user_input) == correct_ans:
                self.score += 1
                print(f"🌟 答對了！太厲害了！目前獲得：{self.score} 分")
                speak("答對了") # 🌟 修改：語音簡化，俐落不囉嗦
            else:
                print(f"❌ 哎呀，答錯了喔！沒關係，正確答案是：{correct_ans}")
                speak("答錯了") # 🌟 修改：語音簡化，俐落不囉嗦

            if self.question_count >= self.max_questions:
                print("🎈 這是最後一題了！即將為你結算總分...")
                time.sleep(1.5) # 稍微縮短等待時間，讓節奏更順
                self.end_game()
            else:
                print("⏳ 準備進入下一題...")
                time.sleep(1.5)
                self.generate_question()

    def show_hint1(self, b=None):
        with self.out_hint1:
            clear_output()
            # 🌟 這裡修改為「往上念」，同步解決文字顯示與語音發音問題
            if self.operator == '+': text = f"試著把較大的數字放在心裡，往上念 {min(self.current_a, self.current_b)}。"
            elif self.operator == '-': text = f"減法是加法的相反！想一想：{self.current_b} 加上多少會等於 {self.current_a}？"
            elif self.operator == '*': text = f"乘法就是連加喔！"
            elif self.operator == '/': text = f"除法是乘法的相反！想一想：{self.current_b} 乘以多少會等於 {self.current_a}？"
            print(f"💡 {text}")
            speak(text)

    def show_hint2(self, b=None):
        with self.out_hint2:
            clear_output()
            if self.operator == '-': text = "可以拿出手指頭，或是畫圈圈來輔助計算喔！"
            elif self.operator == '/': text = f"背一下 {self.current_b} 的九九乘法表吧！"
            else: text = "繼續加油，你可以算出來的！"
            print(f"💡 {text}")
            speak(text)

    def show_ans(self, b=None):
        with self.out_ans:
            clear_output()
            if self.operator == '+': ans = self.current_a + self.current_b
            elif self.operator == '-': ans = self.current_a - self.current_b
            elif self.operator == '*': ans = self.current_a * self.current_b
            elif self.operator == '/': ans = self.current_a // self.current_b
            text = f"正確答案是：{ans}"
            print(f"✅ {text}")
            speak(text)


# ==========================================
# 3. 語文遊戲類別 (LanguageGame)
# ==========================================
class LanguageGame(BaseGame):
    def __init__(self, title, json_file='questions.json', menu_cb=None):
        super().__init__(title, max_questions=0, game_type="國語遊戲", menu_cb=menu_cb)
        self.json_file = json_file
        self.questions_bank = []
        self.shuffled_questions = []
        self.current_question_data = None
        self.answer_input.placeholder = "請輸入選項 (1-4)"
        self._load_questions()

    def _load_questions(self):
        try:
            with open(self.json_file, 'r', encoding='utf-8') as f:
                self.questions_bank = json.load(f)
        except FileNotFoundError:
            self.questions_bank = questions_data
        self.max_questions = len(self.questions_bank)

    def start_game(self):
        if len(self.questions_bank) > 0:
            self.shuffled_questions = random.sample(self.questions_bank, len(self.questions_bank))
        super().start_game()

    def generate_question(self):
        if self.question_count >= self.max_questions or not self.shuffled_questions:
            self.end_game()
            return
        self.current_question_data = self.shuffled_questions[self.question_count]
        self.question_count += 1
        self.is_answered = False

        options_text = "<br>".join(self.current_question_data['options'])
        self.question_html.value = f"<h3 style='color: #2980b9;'>▶️ 第 {self.question_count} / {self.max_questions} 題：<br>{self.current_question_data['q']}</h3><p style='font-size: 16px; margin-left: 20px; line-height: 1.8;'>{options_text}</p>"
        self.answer_input.value = ""
        self._clear_all_outputs()
        with self.out_main: print("✨ 新題目來囉！請輸入選項數字。")

        spoken_options = "，".join([opt.replace('.', ' ') for opt in self.current_question_data['options']])
        speak_text = f"第{self.question_count}題，{self.current_question_data['q']}。選項有：{spoken_options}。"
        speak(speak_text)

    def check_answer(self, b=None):
        with self.out_main:
            clear_output()
            if self.is_answered: return
            user_input = self.answer_input.value.strip()

            if user_input not in ["1", "2", "3", "4"]:
                print("⚠️ 請輸入正確的選項數字 (1, 2, 3 或 4) 喔！")
                speak("請輸入選項")
                return

            self.is_answered = True
            if user_input == self.current_question_data['ans']:
                self.score += 1
                print(f"🌟 答對了！太厲害了！目前獲得：{self.score} 分")
                speak("答對了") # 🌟 修改：語音簡化，俐落不囉嗦
            else:
                print(f"❌ 哎呀，答錯了喔！沒關係，正確答案是選項：{self.current_question_data['ans']}")
                speak("答錯了") # 🌟 修改：語音簡化，俐落不囉嗦

            if self.question_count >= self.max_questions:
                print("🎈 這是最後一題了！即將為解算總分...")
                time.sleep(1.5)
                self.end_game()
            else:
                print("⏳ 準備進入下一題...")
                time.sleep(1.5)
                self.generate_question()

    def show_hint1(self, b=None):
        if not self.current_question_data: return
        with self.out_hint1:
            clear_output()
            text = self.current_question_data['hint1']
            print(f"💡 {text}")
            speak(text)

    def show_hint2(self, b=None):
        if not self.current_question_data: return
        with self.out_hint2:
            clear_output()
            text = self.current_question_data['hint2']
            print(f"💡 {text}")
            speak(text)

    def show_ans(self, b=None):
        if not self.current_question_data: return
        with self.out_ans:
            clear_output()
            text = f"正確答案是選項：{self.current_question_data['ans']}"
            print(f"✅ {text}")
            speak(text)


# ==========================================
# 4. 主選單與左右分割佈局系統
# ==========================================
def setup_application():
    clear_output()

    game_area = widgets.VBox(layout=widgets.Layout(width='68%', margin='5px 1%'))
    game_area.add_class('game-box')

    menu_area = widgets.VBox(layout=widgets.Layout(width='28%', padding='20px', margin='5px 1%'))
    menu_area.add_class('menu-box')

    def show_welcome():
        welcome_html = widgets.HTML(
            "<div style='text-align: center; padding: 40px 10px;'>"
            "<h1 style='color: #2c3e50; font-size: 32px; margin-bottom: 5px;'>"
            "🎮 <ruby>國<rt>ㄍㄨㄛˊ</rt></ruby><ruby>數<rt>ㄕㄨˋ</rt></ruby>小遊戲挑戰賽"
            "</h1>"
            "<h2 style='color: #7f8c8d; font-size: 22px; margin-top: 0;'>(一位數)</h2>"
            "<hr style='width: 60%; border-top: 3px dashed #bdc3c7; margin: 30px auto;'>"
            "<h3 style='color: #34495e; font-size: 24px;'>👈 歡迎遊玩！</h3>"
            "<p style='font-size: 18px; color: #7f8c8d; line-height: 1.6;'>"
            "請從右側選單挑選一個關卡<br>開始你的挑戰吧！🚀"
            "</p>"
            "</div>"
        )
        game_area.children = [welcome_html]

    show_welcome()

    title_ui = widgets.HTML(
        "<div style='text-align: center; margin-bottom: 20px; padding-bottom: 15px; border-bottom: 2px dashed #e0e0e0;'>"
        "<h2 style='margin:0; color: #34495e;'>📌 關卡選單</h2>"
        "</div>"
    )

    btn_layout = widgets.Layout(width='80px', height='80px', margin='10px')

    btn_add = widgets.Button(description="➕ 加法", button_style="success", layout=btn_layout)
    btn_sub = widgets.Button(description="➖ 減法", button_style="danger", layout=btn_layout)
    btn_mul = widgets.Button(description="✖️ 乘法", button_style="info", layout=btn_layout)
    btn_div = widgets.Button(description="➗ 除法", button_style="primary", layout=btn_layout)
    btn_lang = widgets.Button(description="📖 國語", button_style="warning", layout=btn_layout)

    menu_buttons_box = widgets.HBox(
        [btn_add, btn_sub, btn_mul, btn_div, btn_lang],
        layout=widgets.Layout(flex_flow='wrap', justify_content='center')
    )

    for btn in [btn_add, btn_sub, btn_mul, btn_div, btn_lang]:
        btn.add_class('menu-btn')

    def start_add(b):
        game = MathGame("🎮 加法", '+', menu_cb=show_welcome)
        game_area.children = [game.get_ui()]
        game.start_game()

    def start_sub(b):
        game = MathGame("🎮 減法", '-', menu_cb=show_welcome)
        game_area.children = [game.get_ui()]
        game.start_game()

    def start_mul(b):
        game = MathGame("🎮 乘法", '*', menu_cb=show_welcome)
        game_area.children = [game.get_ui()]
        game.start_game()

    def start_div(b):
        game = MathGame("🎮 除法", '/', menu_cb=show_welcome)
        game_area.children = [game.get_ui()]
        game.start_game()

    def start_lang(b):
        game = LanguageGame("🎮 國小三年級：字音字形", menu_cb=show_welcome)
        game_area.children = [game.get_ui()]
        game.start_game()

    btn_add.on_click(start_add)
    btn_sub.on_click(start_sub)
    btn_mul.on_click(start_mul)
    btn_div.on_click(start_div)
    btn_lang.on_click(start_lang)

    menu_area.children = [title_ui, menu_buttons_box]

    main_layout = widgets.HBox([game_area, menu_area])
    display(main_layout)

# 啟動應用程式
setup_application()